In [26]:
# Imports
import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
import numpy as np

### 1) Directory Structure

In [27]:
load_dotenv()

api_key = os.getenv("API_KEY")
repo_owner = 'encode'
repo_name = 'httpx'

url = "https://api.github.com/graphql"
headers = {
    "Authorization": f"Bearer {api_key}"
}

# Rest of code in this cell is partly AI generated
# "... on Tree" is an Inline fragment. If the object is a Tree, it will fetch the entries.
def get_folder_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Tree {
            entries {
              name
              path
              type
              extension
            }
          }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

all_items = []
# Start at the root of the repository
folders_to_check = ["HEAD:"]

while folders_to_check:
    current_folder = folders_to_check.pop(0)
    data = get_folder_contents(current_folder)
    try:
        entries = data['data']['repository']['object']['entries']
    except (KeyError, TypeError):
        continue
        
    for entry in entries:
        all_items.append({
            'Name': entry['name'],
            'Path': entry['path'],
            'Type': entry['type'], # 'blob' for files, 'tree' for folders
            'Extension': entry.get('extension', '')
        })
        
        # If the item is a folder (tree), queue it up to fetch its contents next
        if entry['type'] == 'tree':
            folders_to_check.append(f"HEAD:{entry['path']}/")

# Dump a sample to json for reference
with open("directory_query.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Found {len(all_items)} total files and folders.")

Found 141 total files and folders.


In [28]:
df = pd.DataFrame(all_items)

# Change type column to something human readible
df['Type'] = np.where(df['Type'] == 'tree', 'folder', 'file')

# Sort alphabetically by path
df_folders = df.sort_values(by='Path').reset_index(drop=True)

print("Created CSV File")
df_folders.head(5)

Created CSV File


,Name,Path,Type,Extension
0,.github,.github,folder,
1,CONTRIBUTING.md,.github/CONTRIBUTING.md,file,.md
2,FUNDING.yml,.github/FUNDING.yml,file,.yml
3,ISSUE_TEMPLATE,.github/ISSUE_TEMPLATE,folder,
4,1-issue.md,.github/ISSUE_TEMPLATE/1-issue.md,file,.md


### 2) Naming Conventions

In [29]:
def get_file_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Blob {
              text
            }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

# We're only extracting Python repos. So get all .py files
df_files = df.loc[df['Extension'] == '.py', ['Path']]

# Prefix with HEAD: for the query
df_files['Path'] = 'HEAD:' + df_files['Path']
files_to_check = df_files.squeeze().tolist()

print(files_to_check)
all_items = []

while files_to_check:
    current_file = files_to_check.pop()
    data = get_file_contents(current_file)
    try:
        raw_text = data['data']['repository']['object']['text']
    except (KeyError, TypeError):
        continue
    
    # Remove the prefix HEAD: from the query
    current_file = current_file[5:]
    all_items.append({
        'Path': current_file,
        'Raw text': raw_text
    })

# Dump a sample to json for reference
with open("file_query.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Found {len(all_items)} total Python files.")

['HEAD:httpx/__init__.py', 'HEAD:httpx/__version__.py', 'HEAD:httpx/_api.py', 'HEAD:httpx/_auth.py', 'HEAD:httpx/_client.py', 'HEAD:httpx/_config.py', 'HEAD:httpx/_content.py', 'HEAD:httpx/_decoders.py', 'HEAD:httpx/_exceptions.py', 'HEAD:httpx/_main.py', 'HEAD:httpx/_models.py', 'HEAD:httpx/_multipart.py', 'HEAD:httpx/_status_codes.py', 'HEAD:httpx/_types.py', 'HEAD:httpx/_urlparse.py', 'HEAD:httpx/_urls.py', 'HEAD:httpx/_utils.py', 'HEAD:tests/__init__.py', 'HEAD:tests/common.py', 'HEAD:tests/concurrency.py', 'HEAD:tests/conftest.py', 'HEAD:tests/test_api.py', 'HEAD:tests/test_asgi.py', 'HEAD:tests/test_auth.py', 'HEAD:tests/test_config.py', 'HEAD:tests/test_content.py', 'HEAD:tests/test_decoders.py', 'HEAD:tests/test_exceptions.py', 'HEAD:tests/test_exported_members.py', 'HEAD:tests/test_main.py', 'HEAD:tests/test_multipart.py', 'HEAD:tests/test_status_codes.py', 'HEAD:tests/test_timeouts.py', 'HEAD:tests/test_utils.py', 'HEAD:tests/test_wsgi.py', 'HEAD:httpx/_transports/__init__.py

In [30]:
# Migrate to Pandas Dataframe
df_files = pd.DataFrame(all_items)

# Merge with the original folders dataframe from Section 1.
df_folder_and_files = pd.merge(df_folders, df_files, on='Path')

# CSV files don't really work with rawtext bc of commas so save as a parquet instead
# IMPORTANT: Make sure to run pip install pyarrow
df_folder_and_files.to_parquet('dataset.parquet')

# Final dataframe for Section 2.
df_folder_and_files.head(5)

,Name,Path,Type,Extension,Raw text
0,__init__.py,httpx/__init__.py,file,.py,"from .__version__ import __description__, __ti..."
1,__version__.py,httpx/__version__.py,file,.py,"__title__ = ""httpx""\n__description__ = ""A next..."
2,_api.py,httpx/_api.py,file,.py,from __future__ import annotations\n\nimport t...
3,_auth.py,httpx/_auth.py,file,.py,from __future__ import annotations\n\nimport h...
4,_client.py,httpx/_client.py,file,.py,from __future__ import annotations\n\nimport d...


In [31]:
import ast

# The code written below was referenced in part by AI.

class IdentifierExtractor(ast.NodeVisitor):
    def __init__(self):
        self.defined_classes = set()
        self.defined_funcs = set()
        self.defined_params = set()
        self.defined_vars = set()

    def visit_ClassDef(self, node):
        self.defined_classes.add(node.name)
        self.generic_visit(node)

    def visit_FunctionDef(self, node):
        # Ignore magic methods and test functions
        if not (node.name.startswith('__') and node.name.endswith('__')) and not node.name.startswith('test_'):
            self.defined_funcs.add(node.name)
        self.generic_visit(node)
        
    def visit_AsyncFunctionDef(self, node):
        if not (node.name.startswith('__') and node.name.endswith('__')) and not node.name.startswith('test_'):
            self.defined_funcs.add(node.name)
        self.generic_visit(node)

    def visit_arg(self, node):
        # Ignore standard OOP parameters
        if node.arg not in ('self', 'cls'):
            self.defined_params.add(node.arg)
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name):
                self.defined_vars.add(target.id)
        self.generic_visit(node)
        
    def visit_AnnAssign(self, node):
        if isinstance(node.target, ast.Name):
            self.defined_vars.add(node.target.id)
        self.generic_visit(node)

def extract_codebase_identifiers(source_code) -> dict:
    empty_result = {
        'classes': [], 'functions': [], 'parameters': [], 'variables': []
    }
    
    if not isinstance(source_code, str) or not source_code.strip():
        return empty_result
        
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return empty_result
        
    extractor = IdentifierExtractor()
    extractor.visit(tree)
    
    return {
        'classes': list(extractor.defined_classes),
        'functions': list(extractor.defined_funcs),
        'parameters': list(extractor.defined_params),
        'variables': list(extractor.defined_vars)
    }

In [32]:
extracted_names = df_folder_and_files['Raw text'].apply(extract_codebase_identifiers).apply(pd.Series)

df_folder_and_files = pd.concat([df_folder_and_files, extracted_names], axis=1)

df_folder_and_files.head(5)

,Name,Path,Type,Extension,Raw text,classes,functions,parameters,variables
0,__init__.py,httpx/__init__.py,file,.py,"from .__version__ import __description__, __ti...",[],[main],[],"[__locals, __all__]"
1,__version__.py,httpx/__version__.py,file,.py,"__title__ = ""httpx""\n__description__ = ""A next...",[],[],[],"[__title__, __description__, __version__]"
2,_api.py,httpx/_api.py,file,.py,from __future__ import annotations\n\nimport t...,[],"[put, request, delete, patch, head, options, p...","[trust_env, content, timeout, verify, follow_r...",[__all__]
3,_auth.py,httpx/_auth.py,file,.py,from __future__ import annotations\n\nimport h...,"[Auth, BasicAuth, _DigestAuthChallenge, Functi...","[_resolve_qop, async_auth_flow, _get_client_no...","[username, challenge, nonce, request, file, fu...","[header_value, flow, realm, s, QUOTED_TEMPLATE..."
4,_client.py,httpx/_client.py,file,.py,from __future__ import annotations\n\nimport d...,"[Client, ClientState, BaseClient, BoundAsyncSt...","[send, _merge_url, cookies, _init_transport, _...","[location, verify, max_redirects, other, exc_v...","[location, client_headers, elapsed, next_reque..."


### 3) Type Annotation Coverage
Measures how thoroughly functions use Python type hints. For each `.py` file we count:
- **Parameters with annotations** vs total parameters (excluding `self`/`cls`)
- **Functions with return type annotations** vs total functions

This covers regular args, positional only, keyword only, `*args`, and `**kwargs`.

In [33]:
import ast

def extract_type_annotation_coverage(source_code):
    """Parse Python source and compute type annotation coverage per file."""
    if not isinstance(source_code, str) or not source_code.strip():
        return {
            'total_functions': 0, 'total_params': 0, 'typed_params': 0,
            'functions_with_return_type': 0,
            'param_annotation_pct': 0.0, 'return_annotation_pct': 0.0
        }
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return {
            'total_functions': 0, 'total_params': 0, 'typed_params': 0,
            'functions_with_return_type': 0,
            'param_annotation_pct': 0.0, 'return_annotation_pct': 0.0
        }

    total_functions = 0
    total_params = 0
    typed_params = 0
    functions_with_return_type = 0

    for node in ast.walk(tree):
        if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            continue
        total_functions += 1

        if node.returns is not None:
            functions_with_return_type += 1

        # Regular args (skip self/cls)
        for arg in node.args.args:
            if arg.arg in ('self', 'cls'):
                continue
            total_params += 1
            if arg.annotation is not None:
                typed_params += 1

        # Positional-only and keyword-only args
        for arg in node.args.posonlyargs + node.args.kwonlyargs:
            total_params += 1
            if arg.annotation is not None:
                typed_params += 1

        # *args
        if node.args.vararg:
            total_params += 1
            if node.args.vararg.annotation is not None:
                typed_params += 1

        # **kwargs
        if node.args.kwarg:
            total_params += 1
            if node.args.kwarg.annotation is not None:
                typed_params += 1

    return {
        'total_functions': total_functions,
        'total_params': total_params,
        'typed_params': typed_params,
        'functions_with_return_type': functions_with_return_type,
        'param_annotation_pct': round(typed_params / total_params * 100, 1) if total_params > 0 else 0.0,
        'return_annotation_pct': round(functions_with_return_type / total_functions * 100, 1) if total_functions > 0 else 0.0,
    }

# Apply to each Python file and expand into new columns
type_metrics = df_folder_and_files['Raw text'].apply(extract_type_annotation_coverage).apply(pd.Series)
df_folder_and_files = pd.concat([df_folder_and_files, type_metrics], axis=1)

df_folder_and_files[['Path', 'total_functions', 'total_params', 'typed_params',
                      'param_annotation_pct', 'return_annotation_pct']]

,Path,total_functions,total_params,typed_params,param_annotation_pct,return_annotation_pct
0,httpx/__init__.py,1.0,0.0,0.0,0.0,100.0
1,httpx/__version__.py,0.0,0.0,0.0,0.0,0.0
2,httpx/_api.py,9.0,112.0,112.0,100.0,100.0
3,httpx/_auth.py,19.0,28.0,28.0,100.0,100.0
4,httpx/_client.py,81.0,348.0,348.0,100.0,100.0
5,httpx/_config.py,11.0,17.0,17.0,100.0,100.0
6,httpx/_content.py,17.0,20.0,20.0,100.0,100.0
7,httpx/_decoders.py,31.0,15.0,15.0,100.0,100.0
8,httpx/_exceptions.py,13.0,11.0,11.0,100.0,100.0
9,httpx/_main.py,14.0,46.0,46.0,100.0,100.0


### 4) Docstring Density
Measures documentation coverage at the code level. For each `.py` file we check:
- **Module-level** docstring (the file itself)
- **Every function, async function, and class** definition

A docstring is detected as the first statement being a string constant (`ast.Constant` of type `str`).

In [34]:
def extract_docstring_density(source_code):
    """Parse Python source and compute docstring density per file."""
    if not isinstance(source_code, str) or not source_code.strip():
        return {
            'total_documentable': 0, 'with_docstring': 0,
            'docstring_density_pct': 0.0
        }
    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return {
            'total_documentable': 0, 'with_docstring': 0,
            'docstring_density_pct': 0.0
        }

    def has_docstring(node):
        """Check if an AST node's first statement is a string constant (docstring)."""
        if not hasattr(node, 'body') or len(node.body) == 0:
            return False
        first = node.body[0]
        return (isinstance(first, ast.Expr)
                and isinstance(first.value, ast.Constant)
                and isinstance(first.value.value, str))

    total_documentable = 0
    with_docstring = 0

    # Module-level docstring
    total_documentable += 1
    if has_docstring(tree):
        with_docstring += 1

    # Functions, async functions, and classes
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            total_documentable += 1
            if has_docstring(node):
                with_docstring += 1

    return {
        'total_documentable': total_documentable,
        'with_docstring': with_docstring,
        'docstring_density_pct': round(with_docstring / total_documentable * 100, 1) if total_documentable > 0 else 0.0,
    }

# Apply to each Python file and expand into new columns
doc_metrics = df_folder_and_files['Raw text'].apply(extract_docstring_density).apply(pd.Series)
df_folder_and_files = pd.concat([df_folder_and_files, doc_metrics], axis=1)

df_folder_and_files[['Path', 'total_documentable', 'with_docstring', 'docstring_density_pct']]

,Path,total_documentable,with_docstring,docstring_density_pct
0,httpx/__init__.py,2.0,0.0,0.0
1,httpx/__version__.py,1.0,0.0,0.0
2,httpx/_api.py,10.0,9.0,90.0
3,httpx/_auth.py,26.0,8.0,30.8
4,httpx/_client.py,89.0,49.0,55.1
5,httpx/_config.py,16.0,2.0,12.5
6,httpx/_content.py,22.0,3.0,13.6
7,httpx/_decoders.py,43.0,12.0,27.9
8,httpx/_exceptions.py,42.0,30.0,71.4
9,httpx/_main.py,15.0,1.0,6.7


### 5) Per-Repo Feature Summary
Aggregate the per file metrics into a single row representing the whole repository. This becomes one row in our final feature matrix (one row per repo, one column per feature).

In [35]:
total_params = df_folder_and_files['total_params'].sum()
typed_params = df_folder_and_files['typed_params'].sum()
total_funcs = df_folder_and_files['total_functions'].sum()
funcs_with_return = df_folder_and_files['functions_with_return_type'].sum()
total_documentable = df_folder_and_files['total_documentable'].sum()
with_docstring = df_folder_and_files['with_docstring'].sum()

repo_summary = pd.DataFrame([{
    'repo': f'{repo_owner}/{repo_name}',
    'py_files': len(df_folder_and_files),
    'total_functions': int(total_funcs),

    # Type Annotations
    'param_annotation_coverage_%': round(typed_params / total_params * 100, 1) if total_params > 0 else 0.0,
    'return_annotation_coverage_%': round(funcs_with_return / total_funcs * 100, 1) if total_funcs > 0 else 0.0,

    # Docstring Density
    'docstring_density_%': round(with_docstring / total_documentable * 100, 1) if total_documentable > 0 else 0.0,

    # Directory Structure 
    'max_directory_depth': None,
    'avg_files_per_directory': None,

    # Naming Conventions 
    'pep8_naming_compliance_%': None,
    'avg_identifier_length': None,

    # Test Suite Metrics 
    'test_to_source_ratio': None,
    'test_function_count': None,
    'assertion_density': None,
}])

print(f"=== Feature Summary for {repo_owner}/{repo_name} ===")
repo_summary.T

=== Feature Summary for encode/httpx ===


,0
repo,encode/httpx
py_files,60
total_functions,1134
param_annotation_coverage_%,82.5
return_annotation_coverage_%,53.0
docstring_density_%,23.1
max_directory_depth,None
avg_files_per_directory,None
pep8_naming_compliance_%,None
avg_identifier_length,None


In [36]:
# Re-export with all new feature columns included
df_folder_and_files.to_parquet('dataset.parquet')

### 6) Mutation Previews
Demonstrates how controlled mutations degrade codebase features. Each feature are shown with a before/after example.

---

#### Mutation A: Strip Type Hints + Docstrings 

**Before (original):**
```python
def request(
    method: str,
    url: URL | str,
    params: QueryParamTypes | None = None,
    content: RequestContent | None = None,
    headers: HeaderTypes | None = None,
    timeout: TimeoutTypes = DEFAULT_TIMEOUT_CONFIG,
) -> Response:
    """Sends an HTTP request.

    Args:
        method: HTTP method (GET, POST, etc.)
        url: URL for the request.
        params: Query parameters.
        content: Request body content.
        headers: HTTP headers.
        timeout: Timeout configuration.

    Returns:
        Response object.
    """
    ...
```

**After (type hints + docstring stripped):**
```python
def request(
    method,
    url,
    params=None,
    content=None,
    headers=None,
    timeout=DEFAULT_TIMEOUT_CONFIG,
):
    ...
```

| Metric | Before | After |
|--------|--------|-------|
| param_annotation_coverage | 100% | 0% |
| return_annotation_coverage | 100% | 0% |
| docstring_density | 100% | 0% |

---

#### Mutation B: Flatten Directory Structure 

**Before (organized):**
```
httpx/
├── _client.py
├── _models.py
├── _auth.py
├── _transports/
│   ├── base.py
│   ├── default.py
│   └── asgi.py
└── tests/
    ├── client/
    │   ├── test_client.py
    │   └── test_auth.py
    └── models/
        ├── test_requests.py
        └── test_responses.py
```

**After (flattened):**
```
httpx/
├── _client.py
├── _models.py
├── _auth.py
├── _transports_base.py
├── _transports_default.py
├── _transports_asgi.py
├── test_client.py
├── test_auth.py
├── test_requests.py
└── test_responses.py
```

| Metric | Before | After |
|--------|--------|-------|
| max_directory_depth | 3 | 1 |
| avg_files_per_directory | ~5 | all in root |

---

#### Mutation C: Obfuscate Naming Conventions 

**Before (descriptive):**
```python
def send_request(method: str, url: str) -> Response:
    client = HTTPClient()
    response = client.send(method, url)
    return response
```

**After (obfuscated):**
```python
def fn1(a: str, b: str) -> Response:
    c = HTTPClient()
    d = c.send(a, b)
    return d
```

| Metric | Before | After |
|--------|--------|-------|
| pep8_naming_compliance | 100% | 0% |
| avg_identifier_length | ~12 chars | ~1.5 chars |

---

#### Mutation D: Remove Test Suite 

**Before:**
```
tests/
├── __init__.py
├── conftest.py
├── test_api.py          (14 test functions)
├── test_auth.py         (11 test functions)
├── test_client.py       (49 test functions)
├── client/
│   ├── test_async_client.py
│   └── test_redirects.py
└── models/
    ├── test_requests.py
    └── test_responses.py
```

**After (tests removed):**
```
(empty — all test files deleted)
```

| Metric | Before | After |
|--------|--------|-------|
| test_to_source_ratio | 1.7 (37 test files / 22 source files) | 0 |
| test_function_count | 800+ | 0 |
| assertion_density | ~2.5 per test function | 0 |